In [4]:
import requests
from bs4 import BeautifulSoup
import time

all_urls = []

for page_num in range(1, 10):  # 407 jobs / 50 per page = ~9 pages
    page = requests.get(f'https://job-boards.greenhouse.io/anthropic?page={page_num}', 
                        headers={'User-Agent': 'Mozilla/5.0'})
    soup = BeautifulSoup(page.content, 'lxml')
    
    jobs = soup.find_all('tr', {'class': 'job-post'})
    
    if not jobs:
        print(f'stopped at page {page_num}')
        break
    
    for tr in jobs:
        link = tr.find('td', {'class': 'cell'}).find('a')
        if link:
            all_urls.append(link['href'])
    
    print(f'page {page_num} → total so far: {len(all_urls)}')
    time.sleep(1)

print(f'\ntotal: {len(all_urls)} urls')

page 1 → total so far: 50
page 2 → total so far: 100
page 3 → total so far: 150
page 4 → total so far: 200
page 5 → total so far: 250
page 6 → total so far: 300
page 7 → total so far: 350
page 8 → total so far: 400
page 9 → total so far: 437

total: 437 urls


In [12]:
page = requests.get(all_urls[0], headers={'User-Agent': 'Mozilla/5.0'})
soup = BeautifulSoup(page.content, 'lxml')

desc_div = soup.find('div', {'class': 'job__description body'})

if desc_div:
    for h2 in desc_div.find_all('h2'):
        print(repr(h2.text.strip()))

'About Anthropic'
'What to expect'
'Interview process'
'Compensation'
'Fellows workstreams'
'Across the workstreams, you may be a good fit if you:'
'Strong candidates may also have:'
'Candidates must be:'
'Mentors, research areas, & past projects'
'Unique candidate criteria'
'Mentors, research areas, & past projects'
'Unique candidate criteria'
'Mentors, research areas, & past projects'
'Unique candidate criteria'
'Mentors, research areas, & past projects'
'Unique candidate criteria'
'Mentors, research areas, & past projects'
'Unique candidate criteria'
'Logistics'
"How we're different"
'Come work with us!'


In [15]:
import requests
from bs4 import BeautifulSoup
import csv
import datetime
import time

all_jobs = []

for url in all_urls:
    page = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
    soup = BeautifulSoup(page.content, 'lxml')

    title    = soup.find('h1').text.strip() if soup.find('h1') else None
    desc_div = soup.find('div', {'class': 'job__description body'})

    full_description = ''
    if desc_div:
        for element in desc_div.find_all(['h2', 'h3', 'p', 'li']):
            if element.name in ['h2', 'h3']:
                full_description += f'\n### {element.text.strip()}\n'
            elif element.name == 'p':
                text = element.text.strip()
                if text:
                    full_description += f'{text}\n'
            elif element.name == 'li':
                full_description += f'- {element.text.strip()}\n'

    all_jobs.append({
        'title'            : title,
        'full_description' : full_description.strip(),
        'source_url'       : url,
        'scraped_at'       : datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
    })

    print(f'✓ {title}')
    print(full_description[:200])
    print('---')
    time.sleep(1)

with open('greenhouse_jobs.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['title', 'full_description', 'source_url', 'scraped_at'])
    writer.writeheader()
    writer.writerows(all_jobs)

print(f'\nsaved {len(all_jobs)} jobs to greenhouse_jobs.csv')

✓ Anthropic Fellows Program

### About Anthropic
Anthropic’s mission is to create reliable, interpretable, and steerable AI systems. We want AI to be safe and beneficial for our users and for society as a whole. Our team is a qu
---
✓ Anthropic Fellows Program — AI Safety

### About Anthropic
Anthropic’s mission is to create reliable, interpretable, and steerable AI systems. We want AI to be safe and beneficial for our users and for society as a whole. Our team is a qu
---
✓ Anthropic Fellows Program — AI Security

### About Anthropic
Anthropic’s mission is to create reliable, interpretable, and steerable AI systems. We want AI to be safe and beneficial for our users and for society as a whole. Our team is a qu
---
✓ Anthropic Fellows Program — ML Systems & Performance

### About Anthropic
Anthropic’s mission is to create reliable, interpretable, and steerable AI systems. We want AI to be safe and beneficial for our users and for society as a whole. Our team is a qu
---
✓ Anthropic Fel